1. Read the file using spark dataframe reader API
2. Add metadata columns - Source file, Ingestion timestamp
3. Write to bronze delta table

In [0]:
dbutils.widgets.text("p_batch_id", "")
v_batch_id = dbutils.widgets.get("p_batch_id")

In [0]:
%run ../00-common/01.environment-config

In [0]:
%run ../00-common/02.bronze-helpers

In [0]:
source_file = f"{landing_folder_path}/{v_batch_id}/sprints"
table_name = f"{catalog_name}.{bronze_schema}.sprints"

In [0]:
# Define the schema
sprints_schema = """
    date date,
    raceName string,
    round int,
    season int,
    url string,
    constructorId string,
    driverId string,
    grid int,
    laps int,
    number int,
    points float,
    position int,
    positionText string,
    status string
"""

In [0]:
## Here, we are reading a multiline JSON
sprints_df = (
    spark.read
        .format('json')
        .schema(sprints_schema)
        .option("multiLine", True)
        .option("mode", "FAILFAST")
        .load(source_file)
)

display(sprints_df)

In [0]:
sprints_final_df = add_ingestion_metadata(sprints_df)

display(sprints_final_df)

### Write to bronze delta table

In [0]:
write_to_bronze(
    input_df = sprints_final_df
    , target_table = table_name
    , batch_id = v_batch_id
)

In [0]:
display(spark.table(table_name))

In [0]:
%sql
select season, count(*)
from formula1_incr.bronze.sprints
group by season
order by season